### Widerstand – Lösung Schritt für Schritt

*37. Bundeswettbewerb Informatik (2018), 1. Runde, Aufgabe 5*

Zelda bastelt gerne. Für ihre elektronischen Schaltungen benötigt sie öfters einen bestimmten Widerstand, der nicht in ihrer Grabbelkiste vorhanden ist. Sie kann aber Widerstände kombinieren, die sie schon hat. Zwei Widerstände mit den Werten R1 und R2 können beispielsweise seriell oder parallel geschaltet werden. Dabei entsteht ein neuer Widerstand mit den Werten R1 + R2 (seriell) bzw. 1 : (1 : R1 + 1 : R2) (parallel).

Wendet man das Prinzip mehrfach hintereinander an, kann man zum Beispiel aus Widerständen mit den Werten 1200 Ω, 3300 Ω, 120 Ω und 1000 Ω einen neuen Widerstand mit dem Wert 500 Ω erhalten.

#### Aufgabe 5

Erstelle ein Programm, das zunächst eine Liste von Widerstandswerten – dies sind alle zur Verfügung stehenden Widerstände aus Zeldas Grabbelkiste – und danach den Wert R des von ihr benötigten Widerstands einliest.

Das Programm soll für k = 1, …, 4 eine Schaltung finden, die aus höchstens k Widerständen aus der Grabbelkiste besteht und deren Wert so nah wie möglich an R liegt. Das Programm soll jeweils den Bauplan der Schaltung und ihren Widerstandswert ausgeben.

#### Dateien (`beispieldaten/`)

`widerstaende.txt`: die 24 Widerstände der Grabbelkiste (ein Wert in Ohm pro Zeile). Gesucht sind nacheinander R = 500, 140, 314, 315, 1620, 2719 und 4242 Ω (siehe `readme.md`).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Brute Force (alles ausprobieren):** Es gibt nur wenige Widerstände (24) und höchstens vier davon dürfen benutzt werden. Dann lohnt es sich, **jede** mögliche Schaltung zu bauen und die beste zu behalten. So ist das Ergebnis garantiert optimal.
- **Baum als Datenstruktur:** Eine Schaltung entsteht, indem man immer wieder zwei Teilschaltungen verbindet. Das ist ein Baum: Die Blätter sind Widerstände, jeder innere Knoten ist „in Reihe“ oder „parallel“. Wir speichern einen Baum als **verschachteltes Tupel**, z. B. `('+', 180, 820)`.
- **Rekursion:** Den Wert einer Schaltung und ihre Ausgabe berechnen wir, indem wir die Funktion für die beiden Teilschaltungen aufrufen.
- **Alle Aufteilungen einer Menge:** Eine Schaltung aus mehreren Widerständen entsteht aus zwei Teilschaltungen, die zusammen genau diese Widerstände benutzen. Wir müssen also alle Möglichkeiten durchgehen, die Widerstände in eine linke und eine rechte Gruppe zu teilen.
- **Kleinsten Abstand merken:** Wie bei der Suche nach dem Minimum: Wir merken uns den bisher kleinsten Abstand `abs(wert - R)` und die zugehörige Schaltung.

**Nützliche Funktionen**
- `itertools.combinations(liste, k)`: alle Auswahlen von `k` Elementen (ohne Wiederholung, ohne Beachtung der Reihenfolge).
- `isinstance(x, tuple)`: prüft, ob etwas ein Tupel (also eine Teilschaltung) oder eine Zahl (ein einzelner Widerstand) ist.
- `abs(x)` für den Betrag, `round(x, 6)` zum Runden.

## Die Idee

1. **Zwei Schaltungen verbinden:** Haben wir zwei Teilschaltungen mit den Werten `a` und `b`, gibt es zwei Möglichkeiten: in Reihe (`a + b`) oder parallel (`a · b / (a + b)`; das ist dasselbe wie `1 / (1/a + 1/b)`). Die Reihenfolge von `a` und `b` spielt keine Rolle.
2. **Alle Schaltungen mit einer festen Auswahl von Widerständen:** Für einen einzelnen Widerstand gibt es genau eine Schaltung. Für mehrere teilen wir die Auswahl in eine linke und eine rechte Gruppe, bauen für beide Gruppen rekursiv alle Schaltungen und verbinden jede linke mit jeder rechten, einmal in Reihe und einmal parallel.
3. **Alle Auswahlen durchgehen:** Für k = 1, 2, 3, 4 gehen wir alle Auswahlen von k verschiedenen Widerständen aus der Kiste durch, bauen alle Schaltungen und merken uns für jedes gewünschte `R` die Schaltung mit dem kleinsten Abstand.
4. **„Höchstens k“:** Die beste Schaltung mit höchstens k Widerständen ist die beste unter allen Schaltungen mit 1, 2, ..., k Widerständen.

Wie viele Schaltungen sind das? Bei vier festen Widerständen gibt es 120 verschiedene Schaltungen (die Rechnung steht unten). Mit 24 Widerständen gibt es 10 626 Auswahlen von vier Stück, also rund 1,3 Millionen Schaltungen. Das schafft ein Python-Programm in wenigen Sekunden.

## Schritt 1: Widerstände einlesen

In [ ]:
def lies_widerstaende(dateiname):
    """Liest die Widerstandswerte (ein Wert pro Zeile) und gibt sie als Liste von ganzen Zahlen zurück."""
    with open(dateiname, encoding='utf-8') as datei:
        return [int(zeile) for zeile in datei.read().split()]


widerstaende = lies_widerstaende('beispieldaten/widerstaende.txt')
gesucht = [500, 140, 314, 315, 1620, 2719, 4242]
print(len(widerstaende), 'Widerstände:', widerstaende)

24 Widerstände: [1000, 5600, 180, 6800, 8200, 820, 2700, 150, 100, 1800, 220, 1500, 330, 390, 680, 120, 3900, 3300, 4700, 560, 470, 1200, 270, 2200]


Wichtig: Die Widerstände in der Kiste sind ganze Zahlen, aber Parallelschaltungen ergeben **Kommazahlen**. Wir rechnen deshalb mit `float`-Zahlen. Weil Kommazahlen Rundungsfehler haben können, vergleichen wir unten Abstände mit einer winzigen Toleranz.

## Schritt 2: Eine Schaltung als Baum und ihr Wert

Ein einzelner Widerstand ist einfach seine Zahl, z. B. `180`. Eine Verbindung ist ein Tupel aus dem Zeichen und den beiden Teilschaltungen:

- `('+', a, b)`: `a` und `b` in Reihe,
- `('|', a, b)`: `a` und `b` parallel.

Die Schaltung aus der Aufgabe (1200, 3300, 120, 1000 ergeben 500 Ω) sieht so aus: Zuerst 1200 und 3300 parallel, dazu 120 in Reihe, und das Ganze parallel zu 1000.

In [ ]:
def wert(schaltung):
    """Gesamtwiderstand einer Schaltung (rekursiv)."""
    if not isinstance(schaltung, tuple):          # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = wert(links)
    b = wert(rechts)
    if zeichen == '+':
        return a + b
    else:
        return a * b / (a + b)


beispiel = ('|', ('+', ('|', 1200, 3300), 120), 1000)
print(wert(beispiel))

500.0


Das sind (bis auf Rundung) die 500 Ω aus der Aufgabe: 1200 ∥ 3300 sind etwa 880, plus 120 sind 1000, und zwei parallele 1000er geben 500.

Damit man den Bauplan lesen kann, schreiben wir die Schaltung eingerückt auf. Jede Ebene ist ein Knoten; die zwei Teilschaltungen stehen darunter, weiter rechts. Außerdem gibt es eine Kurzschreibweise mit `+` (Reihe) und `|` (parallel).

In [ ]:
def bauplan(schaltung, einzug=0):
    """Gibt den Bauplan einer Schaltung eingerückt aus."""
    if not isinstance(schaltung, tuple):
        print('  ' * einzug + str(schaltung) + ' Ω')
        return
    zeichen, links, rechts = schaltung
    if zeichen == '+':
        print('  ' * einzug + 'in Reihe (' + str(round(wert(schaltung), 6)) + ' Ω)')
    else:
        print('  ' * einzug + 'parallel (' + str(round(wert(schaltung), 6)) + ' Ω)')
    bauplan(links, einzug + 1)
    bauplan(rechts, einzug + 1)


def als_formel(schaltung):
    """Kurzschreibweise: + für Reihe, | für parallel."""
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'


bauplan(beispiel)
print(als_formel(beispiel))

parallel (500.0 Ω)
  in Reihe (1000.0 Ω)
    parallel (880.0 Ω)
      1200 Ω
      3300 Ω
    120 Ω
  1000 Ω
(((1200 | 3300) + 120) | 1000)


## Schritt 3: Alle Schaltungen aus gegebenen Widerständen

`alle_schaltungen(positionen)` bekommt ein Tupel von **Positionen** in der Kiste (so bleiben gleiche Werte auseinandergehalten) und gibt **alle** Schaltungen zurück, die genau diese Widerstände je einmal benutzen.

- Ist nur ein Widerstand da, gibt es genau eine Schaltung: ihn selbst.
- Sonst teilen wir die Widerstände in eine linke und eine rechte Gruppe. Damit wir jede Aufteilung nur **einmal** betrachten (links und rechts vertauschen ändert nichts), kommt der **erste** Widerstand immer nach links. Aus den übrigen wählen wir mit `combinations` aus, welche noch nach links kommen (keiner, einer, ...; aber nicht alle, sonst wäre rechts nichts mehr).
- Für jede linke und jede rechte Schaltung entstehen zwei neue: in Reihe und parallel.

In [ ]:
from itertools import combinations


def alle_schaltungen(positionen):
    """Alle Schaltungen, die genau die Widerstände an den angegebenen Positionen (Tupel) je einmal benutzen."""
    if len(positionen) == 1:
        return [widerstaende[positionen[0]]]
    erster = positionen[0]
    uebrige = positionen[1:]
    ergebnis = []
    for anzahl in range(0, len(uebrige)):                  # so viele weitere kommen nach links (mindestens einer bleibt rechts)
        for dazu in combinations(uebrige, anzahl):
            links = (erster,) + dazu
            rechts = tuple(p for p in uebrige if p not in dazu)
            for a in alle_schaltungen(links):
                for b in alle_schaltungen(rechts):
                    ergebnis.append(('+', a, b))
                    ergebnis.append(('|', a, b))
    return ergebnis


# Zur Probe: 1 bis 4 Widerstände (die ersten Positionen der Kiste)
for k in range(1, 5):
    print(k, 'Widerstände:', len(alle_schaltungen(tuple(range(k)))), 'Schaltungen')

1 Widerstände: 1 Schaltungen
2 Widerstände: 2 Schaltungen
3 Widerstände: 12 Schaltungen
4 Widerstände: 120 Schaltungen


Das sind 1, 2, 12 und 120 Schaltungen. Zum Nachrechnen: Bei zwei Widerständen gibt es 2 (Reihe oder parallel). Bei dreien wählen wir einen aus (3 Möglichkeiten), schalten ihn mit einer der 2 Schaltungen der anderen beiden zusammen und das in 2 Arten: 3 · 2 · 2 = 12. Bei vieren geben die Aufteilungen 1 + 3 (4 Möglichkeiten, je 12 Schaltungen, 2 Arten) und 2 + 2 (3 Möglichkeiten, je 2 · 2 Schaltungen, 2 Arten) zusammen 4 · 12 · 2 + 3 · 4 · 2 = 120.

Ein paar Schaltungen aus den drei Widerständen 100, 220 und 180:

In [ ]:
for s in alle_schaltungen((8, 10, 2))[:4]:
    print(als_formel(s), '=', round(wert(s), 3))

(100 + (220 + 180)) = 500
(100 | (220 + 180)) = 80.0
(100 + (220 | 180)) = 199.0
(100 | (220 | 180)) = 49.749


## Schritt 4: Die beste Schaltung suchen

Jetzt gehen wir für jedes k alle Auswahlen von k verschiedenen Positionen durch, bauen alle Schaltungen und prüfen sie für **alle** gesuchten Werte `R` gleichzeitig, damit wir das Aufbauen nur einmal machen müssen. In `beste[r]` merken wir uns pro `r` das Paar `(Abstand, Schaltung)`. Nach jedem k halten wir fest, was mit höchstens k Widerständen am besten war (`pro_k`).

Beim Vergleich zählt nur ein **echt kleinerer** Abstand (mit winziger Toleranz gegen Rundungsfehler), damit bei Gleichstand die zuerst gefundene Schaltung bleibt.

In [ ]:
def suche(gesucht_liste):
    """Gibt ein Wörterbuch zurück: pro_k[(R, k)] = (Abstand, Schaltung), die beste Schaltung mit höchstens k Widerständen.
    Zählt außerdem, wie viele Schaltungen geprüft wurden."""
    beste = {}
    pro_k = {}
    geprueft = 0
    for k in range(1, 5):
        for positionen in combinations(range(len(widerstaende)), k):
            for schaltung in alle_schaltungen(positionen):
                geprueft = geprueft + 1
                w = wert(schaltung)
                for r in gesucht_liste:
                    abstand = abs(w - r)
                    if r not in beste or abstand < beste[r][0] - 1e-9:
                        beste[r] = (abstand, schaltung)
        for r in gesucht_liste:
            pro_k[(r, k)] = beste[r]
    return pro_k, geprueft


pro_k, geprueft = suche(gesucht)
print(geprueft, 'Schaltungen geprüft')

1299984 Schaltungen geprüft


Die Zahl passt zur Überlegung oben: 24 + 276 · 2 + 2024 · 12 + 10 626 · 120 = 1 299 984. Nun alle Ergebnisse.

In [ ]:
def zeige(r):
    print('===== R =', r, 'Ω =====')
    for k in range(1, 5):
        abstand, schaltung = pro_k[(r, k)]
        print('k =', k, ':', als_formel(schaltung), '=', round(wert(schaltung), 6), 'Ω')


for r in gesucht:
    zeige(r)

===== R = 500 Ω =====
k = 1 : 470 = 470 Ω
k = 2 : (4700 | 560) = 500.380228 Ω
k = 3 : (1000 | (180 + 820)) = 500.0 Ω
k = 4 : (1000 | (180 + 820)) = 500.0 Ω
===== R = 140 Ω =====
k = 1 : 150 = 150 Ω
k = 2 : (150 | 2200) = 140.425532 Ω
k = 3 : (180 | (820 | 2700)) = 139.949431 Ω
k = 4 : ((180 + (1800 + 120)) | 150) = 140.0 Ω
===== R = 314 Ω =====
k = 1 : 330 = 330 Ω
k = 2 : (6800 | 330) = 314.726508 Ω
k = 3 : ((1800 + 4700) | 330) = 314.055637 Ω
k = 4 : (1000 | (120 + (470 | 1200))) = 313.999343 Ω
===== R = 315 Ω =====
k = 1 : 330 = 330 Ω
k = 2 : (6800 | 330) = 314.726508 Ω
k = 3 : (1800 | (560 | 1200)) = 315.0 Ω
k = 4 : (1800 | (560 | 1200)) = 315.0 Ω
===== R = 1620 Ω =====
k = 1 : 1500 = 1500 Ω
k = 2 : (1500 + 120) = 1620 Ω
k = 3 : (1500 + 120) = 1620 Ω
k = 4 : (1500 + 120) = 1620 Ω
===== R = 2719 Ω =====
k = 1 : 2700 = 2700 Ω
k = 2 : 2700 = 2700 Ω
k = 3 : (1000 + (220 + 1500)) = 2720 Ω
k = 4 : ((180 | 220) + (820 + 1800)) = 2719.0 Ω
===== R = 4242 Ω =====
k = 1 : 3900 = 3900 Ω
k = 2 :

Die Werte stimmen mit der Tabelle der Lösungshinweise überein, zum Beispiel bei R = 500: mit 1 Widerstand 470 Ω, mit 2 Widerständen 500,380228 Ω, mit 3 genau 500 Ω. Bei R = 314 kommt man mit vier Widerständen auf 313,999343 Ω, bei R = 4242 mit vier Widerständen genau auf 4242 Ω. Bei den Schaltungen selbst gibt es oft mehrere gleich gute; die Lösungshinweise zeigen daher teils andere als wir.

Die Aufgabe will einen **Bauplan**. Dafür haben wir `bauplan`. Hier die beste Schaltung für R = 2719:

In [ ]:
abstand, schaltung = pro_k[(2719, 4)]
bauplan(schaltung)

in Reihe (2719.0 Ω)
  parallel (99.0 Ω)
    180 Ω
    220 Ω
  in Reihe (2620 Ω)
    820 Ω
    1800 Ω


Zwei Beobachtungen:

- Erreicht man den Wert schon mit weniger Widerständen (zum Beispiel R = 1620 mit 1500 Ω und 120 Ω in Reihe, also mit 2), steht dasselbe Ergebnis auch bei k = 3 und k = 4. Das ist „höchstens k“.
- Mit mehr Widerständen wird der Abstand nie größer, denn jede Schaltung mit k Widerständen ist auch eine mit höchstens k + 1. Das prüfen wir:

In [ ]:
for r in gesucht:
    abstaende = [pro_k[(r, k)][0] for k in range(1, 5)]
    print(r, [round(a, 6) for a in abstaende])
    assert abstaende == sorted(abstaende, reverse=True)

500 [30, 0.380228, 0.0, 0.0]
140 [10, 0.425532, 0.050569, 0.0]
314 [16, 0.726508, 0.055637, 0.000657]
315 [15, 0.273492, 0.0, 0.0]
1620 [120, 0, 0, 0]
2719 [19, 19, 1, 0.0]
4242 [342, 12, 1.223301, 0.0]


## Für Neugierige: die vierte Stelle mit binärer Suche

Der Aufwand wächst mit der **vierten** Potenz der Zahl der Widerstände (bei 100 Widerständen sind es schon rund 4 Millionen Auswahlen von vier Stück). Die Lösungshinweise zeigen einen Trick: Wir wählen nur **drei** Widerstände aus und lassen den vierten offen. Für die Form „Schaltung aus drei Widerständen, dazu der vierte in Reihe oder parallel“ lässt sich der **ideale** vierte Widerstand ausrechnen. Sei `v` der Wert der Dreierschaltung:

- In Reihe: `v + x = R`, also `x = R - v`.
- Parallel: `1/(1/v + 1/x) = R`, also `x = 1 / (1/R - 1/v)` (nur für `v > R` sinnvoll).

Der Wert der Gesamtschaltung wächst mit `x`, deshalb ist der beste vierte Widerstand einer der beiden **Nachbarn** des idealen Wertes in der sortierten Liste. Wir nehmen zur Sicherheit je zwei Nachbarn, weil einer davon schon in der Dreierschaltung stecken kann. Die Nachbarn findet man mit **binärer Suche** (`bisect.bisect_left`), statt alle übrigen Widerstände zu probieren. `bisect_left(liste, x)` gibt die Position zurück, an der `x` in die sortierte Liste eingefügt würde. Davor und dahinter stehen die Nachbarn.

Die zweite Baumform, zwei Paare `(a ? b) ? (c ? d)`, prüfen wir wie bisher vollständig. Ob eine Dreierschaltung ihren vierten Widerstand mit Trick oder ohne findet, ändert das Ergebnis nicht; wir zählen die geprüften Schaltungen.

In [ ]:
from bisect import bisect_left

# (Wert, Position) sortiert: Tupel werden zuerst nach dem Wert verglichen
paare_sortiert = sorted([(widerstaende[p], p) for p in range(len(widerstaende))])
sortierte_werte = [wert_p for wert_p, p in paare_sortiert]


def suche_schnell(r):
    """Beste Schaltung aus genau 4 Widerständen für r. Gibt (Abstand, Schaltung, Anzahl geprüfter Schaltungen) zurück."""
    beste = (float('inf'), None)
    geprueft = 0
    # Form 1: drei Widerstände zu einer Schaltung, der vierte kommt in Reihe oder parallel dazu
    for drei in combinations(range(len(widerstaende)), 3):
        for s in alle_schaltungen(drei):
            v = wert(s)
            ziele = [('+', r - v)]
            if v > r:
                ziele.append(('|', 1 / (1 / r - 1 / v)))
            for zeichen, x in ziele:
                if x <= 0:
                    continue
                stelle = bisect_left(sortierte_werte, x)
                for nachbar in range(stelle - 2, stelle + 2):      # zwei Kandidaten davor und zwei dahinter
                    if 0 <= nachbar < len(paare_sortiert) and paare_sortiert[nachbar][1] not in drei:
                        schaltung = (zeichen, s, paare_sortiert[nachbar][0])
                        geprueft = geprueft + 1
                        abstand = abs(wert(schaltung) - r)
                        if abstand < beste[0] - 1e-9:
                            beste = (abstand, schaltung)
    # Form 2: zwei Paare
    for vier in combinations(range(len(widerstaende)), 4):
        a, b, c, d = vier
        for paare in [((a, b), (c, d)), ((a, c), (b, d)), ((a, d), (b, c))]:
            for links in alle_schaltungen(paare[0]):
                for rechts in alle_schaltungen(paare[1]):
                    for zeichen in '+|':
                        schaltung = (zeichen, links, rechts)
                        geprueft = geprueft + 1
                        abstand = abs(wert(schaltung) - r)
                        if abstand < beste[0] - 1e-9:
                            beste = (abstand, schaltung)
    return beste[0], beste[1], geprueft


abstand, schaltung, anzahl = suche_schnell(4242)
print(als_formel(schaltung), '=', round(wert(schaltung), 6), 'Ω;', anzahl, 'Schaltungen geprüft')

(((180 | 120) + 3900) + 270) = 4242.0 Ω; 332840 Schaltungen geprüft


Der Abstand ist derselbe wie bei der vollständigen Suche (das prüfen wir am Ende des Generators für alle sieben Werte). Allerdings ist der Gewinn bei 24 Widerständen gering: Die Form „zwei Paare“ wird weiter vollständig durchsucht und macht den größten Teil der Arbeit aus. Der Trick lohnt sich erst bei sehr vielen Widerständen, und dann müsste man auch die zweite Form beschleunigen (zum Beispiel mit einer sortierten Liste aller Paarwerte).

## Zum Schluss: Dasselbe mit MiniZinc

Hier wählt MiniZinc die vier Widerstände **und** die Art der Verbindung selbst. Wir beschreiben nur, wie sich der Gesamtwiderstand aus den Entscheidungen ergibt, und lassen den Abstand zu `R` minimieren.

- `x[1..4]` sind die **Positionen** der vier benutzten Widerstände in der Kiste (`alldifferent`: jeder Widerstand nur einmal). `w[x[i]]` ist der Wert, also eine **Variable als Index** wie in den Arbeitsblättern.
- `reihe[i]` ist `true`, wenn die i-te Verbindung eine Reihenschaltung ist, sonst parallel.
- Es gibt nur **zwei Baumformen** für vier Widerstände: `form = 1` ist eine Kette `((a ? b) ? c) ? d`, `form = 2` sind zwei Paare `(a ? b) ? (c ? d)`. (Die Vertauschungen der Widerstände deckt die freie Wahl von `x` ab.)
- Die Rechnung nutzt **Kommazahlen** (`var float`, `int2float`); dafür braucht man einen Solver, der Gleitkommavariablen unterstützt. Gecode kann das.
- Das Modell benutzt **genau vier** Widerstände. Für „höchstens k“ mit k < 4 müsste man es anpassen. Da unser Python-Programm das schon macht, zeigen wir hier nur k = 4.

Die Modelle sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den [MiniZinc Playground](https://play.minizinc.org/)).

Das Modell (`widerstand.mzn`):

```minizinc
include "alldifferent.mzn";
int: n;                                   % Anzahl der Widerstände in der Kiste
array[1..n] of int: w;                    % ihre Werte
int: R;                                   % gesuchter Wert

array[1..4] of var 1..n: x;               % welche vier Widerstände (Positionen), alle verschieden
array[1..3] of var bool: reihe;           % true: in Reihe, false: parallel
var 1..2: form;                           % 1: ((a ? b) ? c) ? d    2: (a ? b) ? (c ? d)

% verbindet zwei Teilschaltungen in Reihe oder parallel
function var float: verbinde(var bool: in_reihe, var float: a, var float: b) =
  if in_reihe then a + b else a * b / (a + b) endif;

var float: a = int2float(w[x[1]]);
var float: b = int2float(w[x[2]]);
var float: c = int2float(w[x[3]]);
var float: d = int2float(w[x[4]]);
var float: ab = verbinde(reihe[1], a, b);
var float: wert = if form == 1
                  then verbinde(reihe[3], verbinde(reihe[2], ab, c), d)
                  else verbinde(reihe[3], ab, verbinde(reihe[2], c, d)) endif;

constraint alldifferent(x);

var float: abstand = abs(wert - R);
solve minimize abstand;

output [show_float(1, 6, wert), " Ω, Abstand ", show_float(1, 6, abstand), ", Form \(form), ",
        "Reihe \(reihe), Widerstände \([w[x[i]] | i in 1..4])\n"];
```

Die Daten (`.dzn`) erzeugt eine Python-Zelle aus der Kiste:

In [ ]:
def als_dzn(r):
    """Kiste und gesuchter Wert als MiniZinc-Daten (.dzn)."""
    return 'n = ' + str(len(widerstaende)) + ';\nw = ' + str(widerstaende) + ';\nR = ' + str(r) + ';'


print(als_dzn(4242))

n = 24;
w = [1000, 5600, 180, 6800, 8200, 820, 2700, 150, 100, 1800, 220, 1500, 330, 390, 680, 120, 3900, 3300, 4700, 560, 470, 1200, 270, 2200];
R = 4242;


Ergebnisse (Solver Gecode; jeder Wert dauert zwischen 1 und 4 Sekunden):

R = 500:

**Ausgabe:**

```
500.000000 Ω, Abstand 0.000000, Form 1, Reihe [true, false, false], Widerstände [1200, 1800, 1500, 1000]
----------
==========
```

R = 140:

**Ausgabe:**

```
140.000000 Ω, Abstand 0.000000, Form 1, Reihe [true, true, false], Widerstände [120, 1800, 180, 150]
----------
==========
```

R = 314:

**Ausgabe:**

```
313.999343 Ω, Abstand 0.000657, Form 1, Reihe [false, true, false], Widerstände [1200, 470, 120, 1000]
----------
==========
```

R = 315:

**Ausgabe:**

```
315.000000 Ω, Abstand 0.000000, Form 1, Reihe [true, true, false], Widerstände [390, 150, 180, 560]
----------
==========
```

R = 1620:

**Ausgabe:**

```
1620.000000 Ω, Abstand 0.000000, Form 2, Reihe [true, true, false], Widerstände [1200, 1500, 3900, 150]
----------
==========
```

R = 2719:

**Ausgabe:**

```
2719.000000 Ω, Abstand 0.000000, Form 2, Reihe [true, false, true], Widerstände [1800, 820, 220, 180]
----------
==========
```

R = 4242:

**Ausgabe:**

```
4242.000000 Ω, Abstand 0.000000, Form 2, Reihe [true, false, true], Widerstände [270, 3900, 120, 180]
----------
==========
```

Der Abstand 0 bedeutet: genau getroffen. Nur bei R = 314 gibt es mit vier Widerständen keinen exakten Treffer; der Abstand 0,000657 passt zu den 313,999343 Ω in der Tabelle der Lösungshinweise. Die Abstände stimmen mit denen unseres Python-Programms überein; die Schaltungen können sich unterscheiden, weil es oft mehrere gleich gute gibt.

Grenzen des Modells: Es betrachtet nur Schaltungen mit **genau** vier Widerständen und rechnet mit Gleitkommazahlen (Rundung!). Das Python-Programm ist hier flexibler (alle k, Ausgabe als Baum). MiniZinc ist praktisch, wenn die Aufgabe um Bedingungen erweitert wird, etwa „höchstens zwei parallele Teile“.